In [1]:
import pandas as pd
import numpy as np

In [2]:
user_longitude = 12212112.1
user_latitude = 32321212.1
user_altitude = 670

In [3]:
water_level_path = './OutPutData/water_level.xlsx'
water_info_path = './OriginData/waterinfo_data.xlsx'

In [4]:
water_level = pd.read_excel(water_level_path)
water_info = pd.read_excel(water_info_path)
water_level = water_level.drop('序号', axis=1)
water_info = water_info.drop('序号', axis=1)

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances

def process_geospatial_data_multiple_groups(user_lon, user_lat, user_elev, water_inflow_df, water_level_df):
    """
    处理地理空间数据，支持将水位表中所有组的信息添加到结果中
    
    参数:
    user_lon: 用户输入的经度
    user_lat: 用户输入的纬度
    user_elev: 用户输入的高程
    water_inflow_df: 涌水量表DataFrame
    water_level_df: 水位信息表DataFrame
    
    返回:
    整合后的单行DataFrame，包含所有水位组的信息
    """
    # 创建用户输入数据的DataFrame
    user_data = pd.DataFrame({
        '用户经度': [user_lon],
        '用户纬度': [user_lat],
        '用户高程': [user_elev]
    })
    
    # 步骤1: 在涌水量表中找到与用户输入点最近的点
    # 计算每个点到用户输入点的欧氏距离
    user_point = np.array([[user_lon, user_lat, user_elev]])
    water_inflow_points = water_inflow_df[['经度', '纬度', '高程']].values
    
    # 正确计算每个点到用户输入点的距离
    water_inflow_df['距离'] = euclidean_distances(water_inflow_points, user_point).flatten()
    
    # 按距离和日期排序，找到最近的点
    nearest_inflow = water_inflow_df.sort_values(by=['距离', '日期'], ascending=[True, False]).iloc[0]
    
    # 提取最近点的日降雨量和涌水量
    user_data['日降雨量'] = nearest_inflow['日降雨量']
    user_data['涌水量'] = nearest_inflow['涌水量']
    
    # 步骤2: 按经纬度高程分组处理水位信息表
    water_level_groups = water_level_df.groupby(['经度', '纬度', '高程'])
    
    # 遍历每个组并添加到结果中
    for i, (group_key, group_data) in enumerate(water_level_groups):
        group_lon, group_lat, group_elev = group_key
        
        # 计算用户输入点与当前组的欧氏距离
        group_point = np.array([[group_lon, group_lat, group_elev]])
        distance = euclidean_distances(user_point, group_point)[0, 0]
        
        # 获取当前组的最新水位
        latest_water_level = group_data.sort_values('日期', ascending=False).iloc[0]['水位']
        
        # 添加当前组的信息到结果
        user_data[f'水位组{i+1}_经度'] = group_lon
        user_data[f'水位组{i+1}_纬度'] = group_lat
        user_data[f'水位组{i+1}_高程'] = group_elev
        user_data[f'水位组{i+1}_距离'] = distance
        user_data[f'水位组{i+1}_水位'] = latest_water_level
    
    return user_data

In [6]:
print("涌水量表示例数据（前5行）:")
print(water_info.head())

print("\n水位信息表示例数据:")
print(water_level.head())

# 处理数据
result = process_geospatial_data_multiple_groups(user_longitude, user_latitude, user_altitude, water_info, water_level)

print("\n处理结果:")
result

涌水量表示例数据（前5行）:
          日期          经度           纬度   高程   日降雨量      涌水量
0 2023-01-01  400093.254  3044617.466  340  1.747  201.500
1 2023-01-02  400093.254  3044617.466  340  3.160  201.125
2 2023-01-03  400093.254  3044617.466  340  0.857  200.750
3 2023-01-04  400093.254  3044617.466  340  1.712  200.375
4 2023-01-05  400093.254  3044617.466  340  1.942  200.000

水位信息表示例数据:
          日期           经度          纬度      高程       水位
0 2023-03-31  3044699.547  401076.749  905.05  865.482
1 2023-04-01  3044699.547  401076.749  905.05  866.392
2 2023-04-02  3044699.547  401076.749  905.05  867.302
3 2023-04-03  3044699.547  401076.749  905.05  868.212
4 2023-04-04  3044699.547  401076.749  905.05  869.122

处理结果:


C:\Users\Administrator\AppData\Local\Temp\ipykernel_21196\3791716762.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  user_data[f'水位组{i+1}_纬度'] = group_lat
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21196\3791716762.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  user_data[f'水位组{i+1}_高程'] = group_elev
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21196\3791716762.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perf

,用户经度,用户纬度,用户高程,日降雨量,涌水量,水位组1_经度,水位组1_纬度,水位组1_高程,水位组1_距离,水位组1_水位,...,水位组25_经度,水位组25_纬度,水位组25_高程,水位组25_距离,水位组25_水位,水位组26_经度,水位组26_纬度,水位组26_高程,水位组26_距离,水位组26_水位
0,12212112.1,32321212.1,670,4.495,20.0,3044699.547,401076.749,905.05,3.321049e+07,853.476,...,304537741.0,40127407.0,670.61,2.924298e+08,678.44,3.045730e+09,398843.511,941.53,3.033686e+09,910.035
